__<h1 style="text-align: center;font-size: 3rem">Feature Engineering</h1><h2 style="text-align: center;font-size: 1.3rem">(Notebook II)</h2>__

In [18]:
import numpy as np
import pandas as pd
from dotenv import load_dotenv
from os import getenv

from fraud_detection_pipeline.utils.constants import (
    PARENT_DIR,
    RAW_DIRECTORY,
    PROCESSED_DIRECTORY,
)
from fraud_detection_pipeline.data_loading import tts_csv

In [19]:
load_dotenv()
RANDOM_STATE = int(getenv("RANDOM_STATE", 0))
TEST_SPLIT_SIZE = float(getenv("TEST_SPLIT_SIZE", 0.3))

In [20]:
transactions = pd.read_csv(PARENT_DIR / RAW_DIRECTORY / "creditcard.csv")

## Minor Column Changes

Column names are converted from their capitalized format to an all lowercase format.

For improved interpretation of the dataset, the _'Class'_ column (renamed to _'class'_ in the step above) is renamed again to _'is_fraud'_.

This helps associate 1 values (positive) to indicate a fraudulent transaction and associate 0 values (negative) to indicate a genuine transaction.

In [21]:
def renamer(name: str) -> str:
    name = str(name).lower().replace(" ", "_")
    match name:
        case "class":
            return "is_fraud"
        case "time":
            return "time_elapsed"
        case _:
            return name


transactions = transactions.rename(columns=renamer)

print("\n".join(transactions.columns))

time_elapsed
v1
v2
v3
v4
v5
v6
v7
v8
v9
v10
v11
v12
v13
v14
v15
v16
v17
v18
v19
v20
v21
v22
v23
v24
v25
v26
v27
v28
amount
is_fraud


## Transforming Existing Features

### Cyclic Time Features

Cyclic time features provide potential insight in routinely rhythm. This can be helpful if fraudulent transactions happen during a certain time of day.

In [22]:
seconds_in_a_day = 60 * 60 * 24

In [23]:
time_elapsed = transactions["time_elapsed"]

In [24]:
print(f"Days: {time_elapsed.max() / (seconds_in_a_day)}")

Days: 1.9999074074074075


The transactions are recorded over a time period of 2 days. Cyclic features would be influenced by the hour of the day, seeing if the time of day may influence whether a transaction is likely to be fraudulent.

In [25]:
transactions["hour_sin"] = np.sin(2 * np.pi * time_elapsed / seconds_in_a_day)
transactions["hour_cos"] = np.cos(2 * np.pi * time_elapsed / seconds_in_a_day)

In [26]:
transactions[["hour_sin", "hour_cos"]].head(10)

,hour_sin,hour_cos
0,0.000000,1.0
1,0.000000,1.0
2,0.000073,1.0
3,0.000073,1.0
4,0.000145,1.0
5,0.000145,1.0
6,0.000291,1.0
7,0.000509,1.0
8,0.000509,1.0
9,0.000654,1.0


## Type Transformations

As the _'time'_ feature (previously _'Time'_) representing the time elapsed since the first transaction is measured in 1 second intervals, the need to store it as a float which is more appropriate for continuous data is irrelevant. The column's values are casted as a 32-bit integer.

The _'is_fraud'_ feature (previously _'Class'_) is either _1_ to indicate a fraudulent transaction or _0_ to indicate a genuine transaction. Storing this binary value as a 64-bit integer is unnecessary so the type is casted to a boolean.

In [27]:
transactions["time_elapsed"] = transactions["time_elapsed"].astype("int64")
transactions["is_fraud"] = transactions["is_fraud"].astype("int32")
print(transactions.dtypes)

time_elapsed      int64
v1              float64
v2              float64
v3              float64
v4              float64
v5              float64
v6              float64
v7              float64
v8              float64
v9              float64
v10             float64
v11             float64
v12             float64
v13             float64
v14             float64
v15             float64
v16             float64
v17             float64
v18             float64
v19             float64
v20             float64
v21             float64
v22             float64
v23             float64
v24             float64
v25             float64
v26             float64
v27             float64
v28             float64
amount          float64
is_fraud          int32
hour_sin        float64
hour_cos        float64
dtype: object


In [28]:
transactions = transactions.drop(columns=["time_elapsed"])

In [29]:
X = transactions.drop(columns=["is_fraud"])
y = transactions["is_fraud"]

In [30]:
X.shape, y.shape

((284807, 31), (284807,))

Unfortunately, since variables v1 to v28 are unknown in their subject and nature, only that they are continuous values that appear to potentially have normalization performed before ingestion. Additional feature transformations cannot be made.

In [31]:
transactions["is_fraud"] = transactions.pop("is_fraud")

In [32]:
transactions.to_csv(
    PARENT_DIR / PROCESSED_DIRECTORY / "creditcard.csv",
    index=False,
)

In [33]:
X_train, X_test, y_train, y_test = tts_csv(
    PARENT_DIR / PROCESSED_DIRECTORY / "creditcard.csv",
    target="is_fraud",
    test_size=TEST_SPLIT_SIZE,
    random_state=RANDOM_STATE,
)

In [34]:
pd.concat([X_train, y_train], axis=1).to_csv(
    PARENT_DIR / PROCESSED_DIRECTORY / "train.csv",
    index=False,
)
pd.concat([X_test, y_test], axis=1).to_csv(
    PARENT_DIR / PROCESSED_DIRECTORY / "test.csv",
    index=False,
)